In [ ]:
%%writefile matrix_mul.cu

// Include input-output library
#include<iostream>

// Include all standard libraries
#include<bits/stdc++.h>

// Include CUDA library
#include<cuda.h>

// Define block size for GPU threads
#define BLOCK_SIZE 16

// Use standard namespace
using namespace std;


// ---------------- INITIALIZE MATRIX ----------------

// Function to fill matrix with random numbers
void initialize_matrix(int *array, int rows, int cols)
{
    // Loop through rows
    for(int i = 0; i < rows; i++)
    {
        // Loop through columns
        for(int j = 0; j < cols; j++)
        {
            // Store random numbers from 0 to 9
            array[i*cols + j] = rand() % 10;
        }
    }
}


// ---------------- PRINT MATRIX ----------------

// Function to display matrix
void print_matrix(int *array, int rows, int cols)
{
    // Loop through rows
    for(int i = 0; i < rows; i++)
    {
        // Loop through columns
        for(int j = 0; j < cols; j++)
        {
            // Print matrix element
            cout << array[i*cols + j] << " ";
        }

        // Move to next line
        cout << endl;
    }
}


// ---------------- CPU MATRIX MULTIPLICATION ----------------

// Function for matrix multiplication using CPU
void matrix_multiplication_cpu(int *a, int *b, int *c,
                               int common, int c_rows, int c_cols)
{
    // Loop through rows of result matrix
    for(int i = 0; i < c_rows; i++)
    {
        // Loop through columns of result matrix
        for(int j = 0; j < c_cols; j++)
        {
            // Store multiplication sum
            int sum = 0;

            // Multiply row and column elements
            for(int k = 0; k < common; k++)
            {
                sum += a[i*common + k] * b[k*c_cols + j];
            }

            // Store final value
            c[i*c_cols + j] = sum;
        }
    }
}


// ---------------- GPU KERNEL FUNCTION ----------------

// __global__ means this function runs on GPU
__global__ void matrix_multiply(int *a, int *b, int *c,
                                int c_rows, int common, int c_cols)
{
    // Calculate row number for each thread
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // Calculate column number for each thread
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Variable to store multiplication result
    int sum = 0;

    // Check whether thread is inside matrix range
    if(col < c_cols && row < c_rows)
    {
        // Perform matrix multiplication
        for(int j = 0; j < common; j++)
        {
            sum += a[row*common + j] * b[j*c_cols + col];
        }

        // Store result in output matrix
        c[c_cols*row + col] = sum;
    }
}


// ---------------- MAIN FUNCTION ----------------

int main()
{
    // Define matrix dimensions
    int A_rows = 3, A_cols = 3;

    // Rows of B should equal columns of A
    int B_rows = A_cols, B_cols = 3;

    // Result matrix dimensions
    int C_rows = A_rows;
    int C_cols = B_cols;

    // Total number of elements
    int A_size = A_rows * A_cols;
    int B_size = B_rows * B_cols;
    int C_size = C_rows * C_cols;


    // CPU matrices
    int *A, *B, *C;

    // GPU matrices
    int *m1, *m2, *result;

    // Allocate CPU memory
    A = new int[A_size];
    B = new int[B_size];
    C = new int[C_size];


    // ---------------- INITIALIZE MATRIX A ----------------

    initialize_matrix(A, A_rows, A_cols);

    cout << "Matrix 1\n";

    // Print matrix A
    print_matrix(A, A_rows, A_cols);


    // ---------------- INITIALIZE MATRIX B ----------------

    initialize_matrix(B, B_rows, B_cols);

    cout << "Matrix 2\n";

    // Print matrix B
    print_matrix(B, B_rows, B_cols);


    // ---------------- ALLOCATE GPU MEMORY ----------------

    // Allocate unified memory for matrix A
    cudaMallocManaged(&m1, A_size * sizeof(int));

    // Allocate unified memory for matrix B
    cudaMallocManaged(&m2, B_size * sizeof(int));

    // Allocate unified memory for result matrix
    cudaMallocManaged(&result, C_size * sizeof(int));


    // ---------------- COPY CPU -> GPU ----------------

    // Copy matrix A from CPU to GPU
    cudaMemcpy(m1, A, A_size * sizeof(int), cudaMemcpyHostToDevice);

    // Copy matrix B from CPU to GPU
    cudaMemcpy(m2, B, B_size * sizeof(int), cudaMemcpyHostToDevice);


    // ---------------- DEFINE GRID AND BLOCK ----------------

    // Define grid dimensions
    dim3 dimGrid((B_cols + BLOCK_SIZE - 1) / BLOCK_SIZE,
                 (A_rows + BLOCK_SIZE - 1) / BLOCK_SIZE);

    // Define block dimensions
    dim3 dimBlock(BLOCK_SIZE, BLOCK_SIZE);


    // ---------------- GPU TIMING START ----------------

    float gpu_elapsed_time;

    // CUDA events for timing
    cudaEvent_t gpu_start, gpu_stop;

    // Create start event
    cudaEventCreate(&gpu_start);

    // Create stop event
    cudaEventCreate(&gpu_stop);

    // Record start time
    cudaEventRecord(gpu_start);


    // ---------------- LAUNCH GPU KERNEL ----------------

    matrix_multiply<<<dimGrid, dimBlock>>>
    (m1, m2, result, C_rows, A_cols, C_cols);


    // Record stop time
    cudaEventRecord(gpu_stop);

    // Wait until GPU finishes
    cudaEventSynchronize(gpu_stop);

    // Calculate GPU elapsed time
    cudaEventElapsedTime(&gpu_elapsed_time,
                         gpu_start, gpu_stop);


    // ---------------- COPY GPU -> CPU ----------------

    // Copy result from GPU to CPU
    cudaMemcpy(C, result,
               C_size*sizeof(int),
               cudaMemcpyDeviceToHost);


    // ---------------- PRINT GPU RESULT ----------------

    cout << "GPU result:\n";

    // Print GPU result matrix
    print_matrix(C, C_rows, C_cols);

    // Print GPU execution time
    cout << "GPU Elapsed time is: "
         << gpu_elapsed_time
         << " ms" << endl;


    // Destroy CUDA events
    cudaEventDestroy(gpu_start);
    cudaEventDestroy(gpu_stop);


    // ---------------- CPU TIMING ----------------

    // Create events again
    cudaEventCreate(&gpu_start);
    cudaEventCreate(&gpu_stop);

    // Record CPU start time
    cudaEventRecord(gpu_start);


    // Perform matrix multiplication using CPU
    matrix_multiplication_cpu(A, B, C,
                              A_cols, C_rows, C_cols);

    // Record CPU stop time
    cudaEventRecord(gpu_stop);

    // Wait for completion
    cudaEventSynchronize(gpu_stop);

    // Calculate CPU execution time
    cudaEventElapsedTime(&gpu_elapsed_time,
                         gpu_start, gpu_stop);


    // ---------------- PRINT CPU RESULT ----------------

    cout << "CPU result:\n";

    // Print CPU result matrix
    print_matrix(C, C_rows, C_cols);

    // Print CPU execution time
    cout << "CPU Elapsed time is: "
         << gpu_elapsed_time
         << " ms" << endl;


    // ---------------- FREE GPU MEMORY ----------------

    cudaFree(m1);
    cudaFree(m2);
    cudaFree(result);


    // Program ended successfully
    return 0;
}

Writing matrix_mul.cu


In [2]:
!nvcc matrix_mul.cu -o matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [3]:
!./matrix_mul

Matrix 1
3 6 7 
5 3 5 
6 2 9 
Matrix 2
1 2 7 
0 9 3 
6 0 6 
GPU result:
45 60 81 
35 37 74 
60 30 102 
GPU Elapsed time is: 110.042 ms
CPU result:
45 60 81 
35 37 74 
60 30 102 
CPU Elapsed time is: 0.0032 ms
